In [6]:
import requests
import pandas as pd

# =========================
# CONFIG
# =========================
MERPIS_API_TOKEN = "Bearer 13|0c2Tk8HsCjzLf2PrLZlrrsHg2GGZkgTbDBetcoTn34b2a8d2"

url = "https://merpis.ptmerahputih.com/api/downtime/2?year=&month=&customer=&tugboat=&barge="

headers = {
    "Accept": "application/json",
    "Authorization": f"Bearer {MERPIS_API_TOKEN}"
}

BATCH_SIZE = 500

# =========================
# REQUEST API
# =========================
response = requests.get(
    url,
    headers=headers,
    timeout=300
)

print("Status Code:", response.status_code)

if response.status_code != 200:
    print(response.text)
    raise Exception("Gagal mengambil data API")

# =========================
# PARSE JSON
# =========================
json_data = response.json()
data = json_data["data"]["data"]

print("Total Raw Data:", len(data))

# =========================
# PROCESS DATA
# =========================
all_rows = []

for start in range(0, len(data), BATCH_SIZE):
    end = start + BATCH_SIZE
    batch = data[start:end]

    print(f"\nProcessing Batch {start} - {min(end, len(data))}")

    batch_rows = []

    for row in batch:
        projectel = row.get("projectel") or {}

        tugboat = projectel.get("tugboatel") or {}
        barge = projectel.get("bargeel") or {}
        customer = projectel.get("customerel") or {}

        batch_rows.append({
            "Year": row.get("year"),
            "Month": row.get("month"),

            "Code": projectel.get("code"),
            "Tugboat": tugboat.get("name"),
            "Barge": barge.get("name"),
            "Customer": customer.get("fullname"),

            "Prorata": row.get("prorata"),
            "Actual Arrived POL": row.get("arrivepol"),

            "Standard SKAB/LHV": row.get("standardskablhv"),
            "SKAB/LHV": row.get("skablhv"),
            "Downtime SKAB/LHV": row.get("downtimeskablhv"),
            "Days SKAB/LHV": pd.to_numeric(row.get("dayskablhv"), errors="coerce"),

            "Department SKAB/LHV": row.get("departmentskablhv"),
            "Category SKAB/LHV": row.get("categoryskablhv"),
            "Notes SKAB/LHV": row.get("notesskablhv"),

            "Standard FAW to POD": row.get("standardfawsailingtopod"),
            "FAW to POD": row.get("fawsailingtopod"),
            "Downtime FAW to POD": row.get("downtimefawsailingtopod"),
            "Days FAW to POD": pd.to_numeric(row.get("daysfawsailingtopod"), errors="coerce"),

            "Department FAW to POD": row.get("departmentfawsailingtopod"),
            "Category FAW to POD": row.get("categoryfawsailingtopod"),
            "Notes FAW to POD": row.get("notesfawsailingtopod")
        })

    print("Batch Rows:", len(batch_rows))
    all_rows.extend(batch_rows)

# =========================
# DATAFRAME
# =========================
df = pd.DataFrame(all_rows)

# =========================
# FORMAT DATE
# =========================
date_columns = [
    "Actual Arrived POL",
    "Standard SKAB/LHV",
    "SKAB/LHV",
    "Standard FAW to POD",
    "FAW to POD"
]

for col in date_columns:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    ).dt.strftime("%d-%m-%Y %H:%M")

# =========================
# FINAL RESULT
# =========================
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", 100)

print("\n==========================")
print("FINAL RESULT")
print("==========================")

print(df)

print("\nTotal Final Rows :", len(df))
print("Total Columns    :", len(df.columns))

print("\n==========================")
print("DATA TYPES")
print("==========================")

print(df.dtypes)

Status Code: 200
Total Raw Data: 834

Processing Batch 0 - 500
Batch Rows: 500

Processing Batch 500 - 834
Batch Rows: 334

FINAL RESULT
     Year Month              Code           Tugboat              Barge  \
0    2026    05  BG.ML03-2026-008         TB. MP 03  BG. MARLIN 330 03   
1    2026    05  BG.ML01-2026-010         TB. MP 05  BG. MARLIN 330 01   
2    2026    05  BG.ADT2-2026-010   TB. JAS POWER 2      BG. ADITAMA 2   
3    2026    05  BG.MP03-2026-006        TB. DMP 03      BG. MP 330 03   
4    2026    05  BG.MP05-2026-008        TB. DMP 07      BG. MP 330 05   
..    ...   ...               ...               ...                ...   
829  2024    10  BG.ML05-2024-003        TB. DMP 01  BG. MARLIN 330 05   
830  2024    10   BG.SG6-2024-004  TB. ENTERPRISE 3   BG. SEAGATE 2506   
831  2024    10   BG.SG6-2024-005  TB. ENTERPRISE 3   BG. SEAGATE 2506   
832  2024    10  BG.TF99-2024-021  TB. RICKY 1600-5      BG. TAUFIK 99   
833  2024    10  BG.TF99-2024-022  TB. RICKY 1600